# JavaScript Debugging Applications

A practical guide to debugging JavaScript applications using breakpoints, VS Code, Chrome DevTools, and strategies for finding and fixing common bugs.

## Table of Contents

1. [Module 01: Basic Debugging Concepts](#module-01-basic-debugging-concepts)
2. [Module 02: Debugging Backend JavaScript with VS Code](#module-02-debugging-backend-javascript-with-vs-code)
3. [Module 03: Debugging Frontend JavaScript with Chrome DevTools](#module-03-debugging-frontend-javascript-with-chrome-devtools)
4. [Module 04: Finding Common Bugs](#module-04-finding-common-bugs)
5. [Module 05: Final Points](#module-05-final-points)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

## Module 01: Basic Debugging Concepts

Introduces what debugging is, when to do it, and the core tools: breakpoints, stepping controls, watching expressions, exceptions, and reading stack traces.

### What Is Debugging?

> **Debug** — verb: to eliminate errors in or malfunctions of. *(Merriam-Webster)*

Even if you don't know what debugging is, you've probably already done it. Unless you've written 100% flawless code.

#### When to Debug

- During the **development stage**
- During the **test stage**
- When the **application is live** (maintenance stage)

#### What Is a Bug?

A **bug** is an error, flaw, failure, or fault in a computer program. Bugs exist in many shapes and sizes throughout all stages of product development and maintenance.

| Severity | Description |
|---|---|
| **Small Problem** | Minor inconvenience, might not require immediate fixing |
| **Big Problem** | Possible huge consequences, needs to be fixed immediately |

Fixing a bug — debugging — can be as simple as changing one line of code, or as complex as restructuring the entire codebase.

#### Best Practices to Make Debugging Easier

- **Great documentation** — keep records of the system's intended behaviour
- **Well-organized code** — clear structure reduces the search space for bugs
- **Unit tests** — automated tests catch regressions and narrow down the source of failures

### Breakpoints

A **breakpoint** is the "pause button" for your code. It pauses execution at a specific line so you can inspect variables and the flow of the program — like pausing a video to inspect a frame.

#### Setting Breakpoints

- Set at the **specific line of code** where you expect things start to go wrong
- When execution reaches that line, the program **pauses**
- While paused, you can **inspect variables** and the call state
- After inspecting, **press play** to continue — or step through line by line

#### Conditional Breakpoints

You can set a breakpoint to only trigger when a **condition is true**. This avoids stopping on every iteration and lets you focus on the interesting cases.

#### Lines of Code

A single "line" in the debugger's sense is:
- A single instruction (ends with `;`)
- A variable declaration
- A function call
- The opening of an if-statement or block (`{`)

### Stepping Through Code

Once paused at a breakpoint, you control execution one step at a time using the debugger controls.

In [ ]:
render_diagram(Path("assets/01-stepping-through-code.png"))

| Control | Behaviour |
|---|---|
| **Step over** | Execute the current line and move to the next; if it calls a function, run the whole function without entering it |
| **Step into** | If the current line calls a function, enter that function and pause at its first line |
| **Step out** | Finish executing the current function and pause at the line that called it |
| **Continue** | Resume normal execution until the next breakpoint (or program end) |

You can also step through code without a breakpoint by using **Continue** to jump between breakpoints.

### Watching Expressions

While paused, you can **watch expressions** — variables, properties, or any valid JavaScript expression. The debugger evaluates and displays them live as you step.

This is especially helpful when:
- Dealing with a suspicious or tricky code snippet
- The behaviour or value of an expression changes while running
- You want to confirm (or rule out) that a particular value is the cause

> Assume you know which expressions are most relevant to watch before you start stepping — this keeps your debugging session focused.

### Exceptions

An **exception** occurs when something goes wrong at runtime — the "unhappy path". Common causes include:
- Dividing by zero
- Connecting to an unavailable database
- Accessing a property on `undefined` or `null`

#### Handling Exceptions

| Type | Description |
|---|---|
| **Unhandled** | Cause the program to crash — must be addressed |
| **Expected** | Require specific handling (e.g. network timeout → retry) |
| **Unexpected** | Should not occur; indicate a coding mistake to fix |

> Unhandled exceptions crash the program. Handling *expected* exceptions is crucial for a resilient application.

### Reading a Stack Trace

When an exception is thrown, it includes a **stack trace** — a list of the functions that were called leading up to the error. Stack traces are like maps: they lead you directly to where the problem occurred.

A stack trace tells you:
- **The error message** — what went wrong
- **Where the error occurred** — the exact file and line number
- **The call chain** — each function that called the next, from the entry point down to the crash

In [ ]:
render_diagram(Path("assets/02-stack-trace-annotated.png"))

Reading this trace top-to-bottom:

```
TypeError: Cannot read property 'name' of undefined
    at getUserName (/path/to/app/user.js:22:19)   ← error occurred here
    at getUserInfo (/path/to/app/user.js:10:13)   ← getUserInfo called getUserName
    at main (/path/to/app/index.js:5:1)           ← main called getUserInfo
```

The trace tells you `getUserName` was called with an `undefined` argument, so `name` could not be read from it. The likely fix: ensure `getUserInfo` validates or correctly retrieves the value it passes to `getUserName`.

### Demo: Skills App

The course uses a **Skills App** — a simple Node.js/Express application — to demonstrate all debugging concepts hands-on.

See [`01-skills-app-demos/`](01-skills-app-demos/) for the full project. The app has:
- A user interface (HTML pages in `public/`)
- A Node.js backend (`server/app.js` with Express routes)
- JSON data storage (`data/`)

To run it:
```bash
cd 01-skills-app-demos
npm install
npm start
```

The repository has multiple branches, each containing a version of the app with a specific bug to debug.

### Module 01 Summary

| Concept | Key Point |
|---|---|
| **Debugging** | Eliminating errors at any stage of development |
| **Breakpoints** | Pause execution at a specific line to inspect state |
| **Step over/into/out** | Navigate code line-by-line with precise control |
| **Watching expressions** | Monitor variable values live while stepping |
| **Exceptions** | Runtime errors; unhandled ones crash the program |
| **Stack traces** | Show the call chain leading to an error |

---

## Module 02: Debugging Backend JavaScript with VS Code

Covers setting up VS Code for Node.js debugging, setting breakpoints, stepping through server-side code, inspecting data, and using logpoints and auto-attach.

### Getting VS Code Ready

**Prerequisites:**
- Node.js installed
- The project folder open in VS Code (required to add a `launch.json`)

#### Two ways to start debugging in VS Code

**Quick start — single file:**
1. Open the Run and Debug view (Ctrl+Shift+D / Cmd+Shift+D)
2. Click "Run and Debug" — VS Code auto-detects Node.js and runs the current file

**Full setup — with `launch.json`:**
1. Open the Run and Debug view
2. Click "create a launch.json file"
3. Select "Node.js" as the environment
4. A `.vscode/launch.json` is created — configure the entry point and any args

Once `launch.json` is configured, the project is ready for repeatable debug sessions.

### Setting Breakpoints in VS Code

Breakpoints pause the application so you can **inspect data** and then continue execution step by step.

To set a breakpoint:
1. Click in the **gutter** (left margin) of the editor next to a line — a red dot appears
2. Start the app in **Run and Debug** mode
3. When execution reaches the breakpoint, VS Code pauses and highlights the line

You can also set a **conditional breakpoint** (right-click the gutter): the debugger only pauses when the condition evaluates to `true`.

### Stepping Through Backend Code

The debugging controls appear at the top of the editor when paused:

| Button | Action |
|---|---|
| Continue (F5) | Resume until next breakpoint |
| Step over (F10) | Execute current line, stay in same function |
| Step into (F11) | Enter the called function |
| Step out (Shift+F11) | Finish current function, return to caller |
| Restart | Restart the debug session |
| Stop | Stop the debug session |

The debugger always follows the **normal execution path** of the program — it never skips lines, it just lets you control the pace.

### Logpoints

A **logpoint** is a variant of a breakpoint that:
- Does **not** pause execution
- Logs a message (with optional expressions) to the Debug Console
- Useful for tracing values without stopping the program

To add a logpoint: right-click the gutter → "Add Logpoint" → enter a message with `{expression}` placeholders.

### Auto Attach

VS Code's **Auto Attach** feature automatically attaches the debugger to Node.js processes you start from the integrated terminal — without needing to configure a `launch.json` or manually start a debug session.

Enable it via the status bar (`Auto Attach: Off` → toggle on).

### Module 02 Summary

VS Code's debugging tools are designed for **server-side and script-level** JavaScript:

- **Run and Debug view** — central hub for debug sessions
- **`launch.json`** — configures repeatable debug sessions per project
- **Breakpoints + stepping** — pause and navigate execution
- **Logpoints** — non-intrusive logging without stopping the program
- **Auto Attach** — automatically debug Node processes from the terminal

---

## Module 03: Debugging Frontend JavaScript with Chrome DevTools

Covers the Chrome DevTools panels used for frontend debugging: Elements, Console, Sources, Network, Performance, Memory, and Application.

### What Is Chrome DevTools?

**Chrome DevTools** is a set of web developer tools built directly into the Chrome browser. Each panel gives insight into a specific part of the app, and you can interact with the live page from within DevTools.

Open with: **F12** or right-click → **Inspect**.

### Elements Panel

The Elements panel is the **gateway to the HTML and CSS** of your application. It shows the live DOM tree and applied styles.

- Primarily used for **debugging HTML and CSS**
- Still important for JavaScript debugging: DOM structure is directly affected by JS and event listeners are visible here
- Changes made in Elements are **live but temporary** — they reset on page reload

### Console Panel

The Console is similar to the debug console in VS Code:

- Displays **errors**, **warnings**, and `console.log()` output
- The **first place to look** when JavaScript on a page isn't working
- Accepts JavaScript input — you can execute expressions against the live page

### Sources Panel

The Sources panel is the primary tool for JavaScript debugging in the browser:

- **View all loaded files** — JS, HTML, CSS, source maps
- **Set breakpoints** on any line of JavaScript
- **Step through execution** using the same controls as VS Code (step over, into, out)
- **Inspect variables and the call stack** while paused

### Network Panel

The Network panel **logs all network activity** in real time:

- Loading times and order of requests
- Request details: method, URL, headers, status code
- Response bodies
- Useful for diagnosing **loading issues, failed requests, and API errors**

### Performance and Memory Panels

#### Performance Panel
Provides granular insight into your app's **speed and efficiency**:
- Records a timeline of everything that happens (scripting, rendering, painting)
- Identifies **runtime performance issues** — slow functions, long tasks, frame drops

#### Memory Panel — Heap Snapshots
Helps identify **memory leaks**:
- Take a **snapshot** of the JavaScript heap at any point in time
- Take multiple snapshots and **compare them** to find objects accumulating unnecessarily
- Identifies which objects are holding memory and preventing garbage collection

### Application Panel

The Application panel is your one-stop shop for inspecting **all data related to your web app**:

- **IndexedDB / Web SQL** — client-side databases
- **Local storage / Session storage** — key-value stores
- **Cookies** — inspect and edit cookies per domain
- Service Workers, Cache Storage, and more

### Module 03 Summary

Chrome DevTools panels and their primary use:

| Panel | Use For |
|---|---|
| **Elements** | Inspect and edit live DOM and CSS |
| **Console** | View errors, warnings, `console.log` output |
| **Sources** | Set breakpoints, step through JS, inspect state |
| **Network** | Monitor requests, responses, loading order |
| **Performance** | Identify runtime slowness and long tasks |
| **Memory** | Detect memory leaks via heap snapshots |
| **Application** | Inspect storage, cookies, service workers |

Chrome DevTools is the **daily go-to tool** for analyzing and solving problems within web applications.

---

## Module 04: Finding Common Bugs

A systematic walkthrough of the most common JavaScript bug categories, with diagnostic approaches for each.

### Reference and Type Errors

| Error Type | Definition |
|---|---|
| **ReferenceError** | Using an identifier that has not been defined |
| **TypeError** | Performing an operation not available on a certain type |

#### Diagnosing Reference Errors
1. Check the **Console** — the error message identifies the undefined identifier
2. Verify the variable is declared and in scope at the point of use
3. In non-strict mode, JavaScript allows undeclared variables — always use `"use strict"` to catch these early

#### Diagnosing Type Errors
1. Check the **Console** — the message shows which operation failed and on what
2. Verify the type of the value at the point of use (use the debugger or `typeof`)
3. Fix the code: either guard against the unexpected type or ensure the correct type is passed

### Logical Errors

Logical errors are the **trickiest to spot**:
- Typically produce **no console errors** — the code runs, but gives wrong output
- Require understanding the intended behaviour to recognise the actual problem

#### Diagnosing Logical Errors
1. Set **breakpoints** at the relevant section
2. Step through and **inspect the code's flow and state** — watch key variables
3. Identify the line where the value diverges from expectation
4. Correct the logic

### Undefined Behavior

**Undefined behavior** arises from gaps in the code's logic — situations the developer didn't account for. The outcome is unpredictable.

#### Diagnosing Undefined Behavior
1. Use the **Elements panel** to inspect event listeners attached to DOM elements
2. Fall back to the **Sources panel** to trace JS execution
3. Fix the gap in the logic

### Browser Compatibility Issues

Some JavaScript APIs are not available in all browsers or browser versions.

```js
const numbers = [1, 2, 3];
console.log(numbers.includes(2)); // not available in very old browsers
```

**Approach:**
- Check MDN or caniuse.com for browser support tables
- Bugs that "don't occur on my computer" are often compatibility issues
- Always **include the browser version** in bug reports — it's critical diagnostic information

### Performance Issues

JavaScript performance problems arise from:
- **Memory leaks** — objects held in memory unnecessarily
- **Inefficient code** — algorithms doing too much work per operation
- **Unnecessary computations** — work that could be avoided or cached

#### Memory Leaks

A **memory leak** occurs when the program holds onto memory that is no longer needed. This happens because objects retain a reachable reference, preventing the garbage collector from reclaiming the memory.

**The garbage collector:**
- An algorithm that frees memory occupied by objects with no remaining references
- In JavaScript, memory allocation is automatic — the GC handles deallocation
- A memory leak = a reference that should have been cleared but wasn't

**Example: accidental long-lived reference**

```js
let data = [];
for (let i = 0; i < 10000; i++) {
    data.push(new Array(1000).join('*'));
}
// ... other code ...
// If 'data' is never cleared, it keeps growing in memory

data = null; // explicitly release the reference so the GC can reclaim it
```

#### Diagnosing Performance Issues
1. Suspect a **memory leak** or an **inefficient function**
2. Use the **Performance panel** to record a session and identify slow operations
3. Use the **Memory panel** to take heap snapshots before and after an action — compare them to find leaking objects

### Page Loading and Timing Issues

Resource-heavy applications or server delays cause loading problems. These affect user experience directly.

**Diagnosing loading issues:**
1. Open the **Network panel** before reproducing the problem
2. **Disable cache** (in Network panel settings) to see real load times
3. Reload and inspect each network request: check status codes, response times, and order
4. Identify the bottleneck (oversized asset, slow endpoint, blocking script) and fix it

### Failed Requests to the Backend

Frontend apps communicate with backend APIs. When this fails:

**Common causes:**
- Wrong URL or endpoint path
- Server error (5xx)
- Authentication failure (401/403)
- CORS issues
- Network connectivity

**Diagnosing failed requests:**
1. **Check the Console** first — network errors often appear there
2. Open the **Network panel** — find the failed request (shown in red)
3. Inspect the request headers, response status code, and response body
4. Fix the root cause (code, server, or configuration)

### Failed Imports

JavaScript code is split into modules. External packages can also be imported. When an import fails, it breaks the whole module that depends on it.

**Common causes:**
- Incorrect file paths
- Module not installed (`npm install` not run)
- Syntax errors inside the imported module

**Diagnosing failed imports:**
1. Check the **Console** — import errors produce clear messages
2. Verify the import path in your code (relative paths and typos are common)
3. Confirm the module is available (`node_modules/` exists, or path is correct)
4. Fix and re-run

### Problems with UI Events

JavaScript listens for user interactions (clicks, input, hover, form submissions) via **event handlers**. When a button or form doesn't respond as expected:

**Common causes:**
- Event listener not attached to the correct element
- Event listener added at the wrong time (before the element exists)
- The handler function has a bug

**Diagnosing UI event problems:**
1. In the **Elements panel**, select the DOM element and check the **Event Listeners** tab
2. Confirm the handler is attached and to the right element
3. Use the **Console** and **Sources panel** to trace what happens when the event fires
4. Fix the code

### Callbacks, Promises, and Async/Await Issues

Asynchronous JavaScript is a common source of bugs because:
- Code doesn't execute top-to-bottom linearly
- The presence and order of values becomes harder to predict
- Errors can be swallowed silently if not handled correctly

#### Callbacks and Promises

- **Callbacks** — functions passed as arguments to be called when an async operation completes
- **Promises** — objects that represent a future value (pending → fulfilled or rejected)

#### async/await

`async/await` is syntactic sugar over Promises that makes async code read like synchronous code — but it can introduce subtle bugs:

| Issue | Description |
|---|---|
| **Unhandled rejections** | A rejected promise with no `.catch()` or `try/catch` |
| **Unresolved promises** | `await` used incorrectly, promise never settles |
| **Sync code interference** | Code before/after `await` runs in unexpected order |

#### Diagnosing Async Issues
1. **Check the Console panel** — unhandled rejections appear as errors
2. Use the **Sources panel** — set breakpoints inside `.then()` blocks or after `await` statements
3. Step through to see when and whether the async operation resolves
4. Fix the bug: add error handling, correct the `await` placement, or fix the promise chain

### Module 04 Summary

| Bug Category | Primary Tool | Key Signal |
|---|---|---|
| Reference / Type errors | Console | Error message in console |
| Logical errors | Sources (breakpoints) | Wrong output, no error |
| Undefined behavior | Elements → Event Listeners | Feature silently does nothing |
| Browser compatibility | MDN / caniuse | Works on my machine only |
| Performance / Memory leaks | Performance + Memory panels | Slowness or growing memory |
| Page loading issues | Network panel | Long load times, blocked resources |
| Failed requests | Console + Network panel | Red requests, 4xx/5xx status |
| Failed imports | Console | Import/module not found error |
| UI event problems | Elements → Event Listeners | Click/input handlers not firing |
| Async/await issues | Console + Sources | Unhandled rejection, wrong order |

Debugging happens throughout the entire software lifecycle. Practice and familiarity with these tools makes each debugging session faster.

---

## Module 05: Final Points

Compares VS Code and Chrome DevTools, and covers best practices that make debugging faster and less frequent.

### VS Code vs Chrome DevTools

Both tools support JavaScript debugging, but they excel in different contexts. There is no single "best pick" — the right tool depends on what you're debugging.

| Tool | Best For |
|---|---|
| **VS Code** | Server-side JavaScript (Node.js), navigating large codebases, IDE-integrated workflow |
| **Chrome DevTools** | Client-side JavaScript, browser-specific issues, analyzing performance, network, and storage |

In practice, a full-stack JavaScript developer will use **both** — VS Code for the backend and Chrome DevTools for the frontend.

### Best Practices for Debugging

Following best practices reduces the frequency of bugs and makes debugging sessions shorter when bugs do occur.

- **Creating and updating documentation** — clear specs reduce misunderstandings that cause bugs
- **Adding meaningful comments** — explain the *why* for non-obvious code decisions
- **Writing test cases and unit tests** — automated tests catch regressions and narrow the search space
- **Code profiling** — regularly profile to catch performance problems before they become critical
- **Collaboration and teamwork** — a fresh pair of eyes often spots what you've been staring past

> Sticking to best practices can speed up debugging and even prevent some bugs from appearing in the first place.

### Module 05 Summary

Every bug is a chance to improve your skills and become a better developer.

- Use **VS Code** for backend/script debugging
- Use **Chrome DevTools** for frontend/browser debugging
- Apply **best practices** (docs, tests, profiling, collaboration) to reduce bug frequency
- Debugging is a skill that improves with practice — keep calm and happy debugging